# Õppetund 05 - Agentne RAG


## Seadistamine

See märkmik demonstreerib Agentic RAG (otsingul põhinev genereerimine) mustrit, kasutades Microsoft Agent Frameworki.

**Nõuded:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — teie Azure AI Search teenuse lõpp-punkt
- `AZURE_SEARCH_API_KEY` — teie Azure AI Search API võti
- Azure OpenAI juurutus seadistatud keskkonnamuutujate kaudu
- Azure CLI autentitud (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Mis on agentne RAG?

Traditsiooniline RAG järgib fikseeritud töövoogu: esmalt tuuakse dokumendid, seejärel genereeritakse vastus. **Agentne RAG** läheb sellest kaugemale, andes agendile autonoomia otsustada, **millal** ja **kuidas** teavet otsida.

Agentse RAG-iga saab agent:
- **Otsustada**, kas vastuse andmiseks on vaja teavet otsida
- **Valida**, millist andmeallikat või tööriista pärida
- **Hinnata** leitud tulemusi ja vajadusel teha järelpäringuid, kui esimene katse on ebapiisav
- **Ühendada** teave mitmest otsingusammust koherentseks vastuseks

See muudab agendi paindlikumaks ja täpsemaks võrreldes staatilise too- ja genereeri töövooga.


## Otsingutööriista loomine

Agentic RAG-is on välistest andmeallikatest tehtud **tööriistad**, mida agent saab vajadusel kutsuda. See võimaldab agendil käsitleda andmete hankimist lihtsalt ühe tegevusena, mitte kohustusliku sammuna.

Allpool määratleme reisiteadmiste baasi ja avaldame selle tööriistana, mida agent saab sihtkoha teabe otsimiseks kasutada.


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## RAG-agenti loomine

Nüüd loome agendi, kes on juhendatud **alati enne vastamist teavet otsima**. Agent kasutab tööriista `search_travel_knowledge`, et oma vastuseid teadmistebaasi põhjal toetada, mitte tugineda vaid oma koolitusandmetele.


In [ ]:
agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## Iteratiivne päring — Maker-Checker muster

Agentic RAG peamine eelis on **iteratiivne päring**. Agent saab teha mitu otsinguringi, et kinnitada, täpsustada või laiendada oma esialgseid leide — sarnaselt "maker-checker" töövoole:

1. **Maker-ettevõtmine**: Agent hangib esialgse teabe ja koostab vastuse kavandi.
2. **Checker-ettevõtmine**: Agent teeb lisapäringuid, et kontrollida üksikasju või täita puudujääke.

Alloleval juhul esitatakse agendile küsimus, mis nõuab mitme sihtkoha võrdlemist, sundides teda korduvateks otsinguteks.


In [ ]:
checker_agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## Kokkuvõte

Selles õppetükis õppisite, kuidas ehitada Microsoft Agent Frameworki abil **Agentic RAG** süsteem:

- **Agentic RAG** võimaldab agendil iseseisvalt otsustada, millal infot otsida, muutes päringu dünaamiliseks, mitte fikseerituks.
- **Tööriistad andmeallikatena**: Välised teadmistebaasid (näiteks Azure AI Search) on mähitud tööriistadeks, mida agent saab kutsuda.
- **Iteratiivne otsing**: maker-checker muster võimaldab agendil teha mitu otsingutsüklit — otsida, kinnitada ja täpsustada — enne lõpliku vastuse genereerimist.

Tootmises asendaksite mälus oleva `TRAVEL_KNOWLEDGE_BASE` päris Azure AI Search indeksiga, et käsitleda suurmahulisemat reisi dokumentide otsimist.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vastutusest loobumine**:
See dokument on tõlgitud kasutades tehisintellektil põhinevat tõlketeenust [Co-op Translator](https://github.com/Azure/co-op-translator). Kuigi püüame tagada täpsust, palun arvestage, et automatiseeritud tõlked võivad sisaldada vigu või ebatäpsusi. Originaaldokument tema emakeeles tuleks pidada usaldusväärseks allikaks. Olulise info puhul soovitatakse kasutada professionaalset inimtõlget. Me ei vastuta ühegi väärarusaama või valesti tõlgendamise eest, mis võib tekkida selle tõlke kasutamisest.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
